In [34]:

import json
from datetime import datetime, timezone
from html import escape
from pathlib import Path


# Descriptions for the WMO weather codes returned by Open-Meteo.
WEATHER_CODE_DESCRIPTIONS = {
    0: "Clear sky",
    1: "Mainly clear",
    2: "Partly cloudy",
    3: "Overcast",
    45: "Fog",
    48: "Depositing rime fog",
    51: "Light drizzle",
    53: "Moderate drizzle",
    55: "Dense drizzle",
    56: "Light freezing drizzle",
    57: "Dense freezing drizzle",
    61: "Slight rain",
    63: "Moderate rain",
    65: "Heavy rain",
    66: "Light freezing rain",
    67: "Heavy freezing rain",
    71: "Slight snowfall",
    73: "Moderate snowfall",
    75: "Heavy snowfall",
    77: "Snow grains",
    80: "Slight rain showers",
    81: "Moderate rain showers",
    82: "Violent rain showers",
    85: "Slight snow showers",
    86: "Heavy snow showers",
    95: "Thunderstorm",
    96: "Thunderstorm with slight hail",
    99: "Thunderstorm with heavy hail",
}


def generate_report(log_path, output_path):
    """Create an HTML weather report from a JSON history log.

    Args:
        log_path: Path to the JSON weather log created in Task 2.
        output_path: Path where the generated HTML report will be saved.
    """
    # Convert the supplied path strings into Path objects for file operations.
    log_file = Path(log_path)
    output_file = Path(output_path)

    # Read the Task 2 log, treating a missing log as an empty history.
    if log_file.exists():
        with log_file.open("r", encoding="utf-8") as file:
            log = json.load(file)
    else:
        log = []

    # Record when the report was created and find each unique city in the log.
    generated_at = datetime.now(timezone.utc).isoformat()
    cities = sorted({entry["city"] for entry in log})

    # Calculate the earliest and latest observation timestamps for the summary.
    if log:
        timestamps = [entry["fetched_at"] for entry in log]
        date_range = f"{min(timestamps)} to {max(timestamps)}"
    else:
        date_range = "No records"

    # Build one HTML table row for every observation in the log.
    rows = []
    for entry in log:
        weather = entry["weather"]
        weather_code = weather["weathercode"]

        # Translate the numeric WMO code into a reader-friendly description.
        description = WEATHER_CODE_DESCRIPTIONS.get(
            weather_code, f"Unknown code ({weather_code})"
        )

        # Escape values before inserting them so special HTML characters are safe.
        rows.append(
            "<tr>"
            f"<td>{escape(str(entry['city']))}</td>"
            f"<td>{escape(str(entry['fetched_at']))}</td>"
            f"<td>{escape(str(weather['temperature']))}</td>"
            f"<td>{escape(str(weather['windspeed']))}</td>"
            f"<td>{escape(description)}</td>"
            "</tr>"
        )

    # Keep the table meaningful when the log contains no observations.
    if not rows:
        rows.append('<tr><td colspan="5">No weather records available.</td></tr>')

    city_text = ", ".join(cities) if cities else "None"

    # Assemble the complete page, including its CSS, summary, and table.
    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Weather Report</title>
  <style>
    body {{ font-family: Arial, sans-serif; margin: 2rem; color: #1f2937; }}
    h1, h2 {{ color: #075985; }}
    .summary {{ background: #f0f9ff; padding: 1rem; border-radius: 0.5rem; }}
    table {{ width: 100%; border-collapse: collapse; margin-top: 1rem; }}
    th, td {{ border: 1px solid #cbd5e1; padding: 0.75rem; text-align: left; }}
    th {{ background: #e0f2fe; }}
    tr:nth-child(even) {{ background: #f8fafc; }}
  </style>
</head>
<body>
  <h1>Weather Report — Generated {escape(generated_at)}</h1>
  <section class="summary">
    <h2>Summary</h2>
    <p><strong>Total records:</strong> {len(log)}</p>
    <p><strong>Cities tracked:</strong> {escape(city_text)}</p>
    <p><strong>Date range:</strong> {escape(date_range)}</p>
  </section>
  <h2>Weather observations</h2>
  <table>
    <thead>
      <tr>
        <th>City</th>
        <th>Timestamp</th>
        <th>Temperature (°C)</th>
        <th>Windspeed (km/h)</th>
        <th>Weather description</th>
      </tr>
    </thead>
    <tbody>
      {''.join(rows)}
    </tbody>
  </table>
</body>
</html>
"""

    # Write the finished self-contained report to the requested output file.
    with output_file.open("w", encoding="utf-8") as file:
        file.write(html)

In [35]:
generate_report("weather_log.jason", "weather_report.html")
